# Security under collapse — Colab (protocol_v2)

Inference only. No training.

**Runtime: T4 GPU.** Do not select A100.

Run cells top to bottom. After the gate cell, stop and read the numbers before P1.

If the session dies: Runtime → reconnect, rerun the setup cell, then rerun the cell that was in progress. Finished work is already on Drive.


In [ ]:
# 1. GPU check. Stop if this is not a T4 (or an L4 you chose on purpose).
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# 2. Mount Drive and enter the uploaded repo.
# The folder must contain scripts/, src/, results/, notebooks/.
from google.colab import drive
drive.mount("/content/drive")

import os
REPO = "/content/drive/MyDrive/collapse_eval"
if not os.path.isdir(os.path.join(REPO, "scripts")):
    raise SystemExit(f"Repo not found at {REPO}. Unzip the upload so scripts/ is inside it, then edit REPO.")
os.chdir(REPO)
get_ipython().run_line_magic("cd", REPO)
print("cwd", os.getcwd())


In [ ]:
# 3. Dependencies. Do not reinstall torch — Colab's CUDA build is the one we want.
!pip -q install "transformers>=4.51.0" "peft>=0.14.0" "datasets>=2.19" accelerate sentencepiece protobuf pyyaml safetensors "langfuse>=3" tqdm


In [ ]:
# 4. Secrets, VM weight cache, and a Drive heartbeat.
# Add OPENROUTER_API_KEY and HF_TOKEN (or HUGGING_FACE_API_KEY) under the key icon.
# Optional: LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY, LANGFUSE_BASE_URL.
import os, threading, time
from pathlib import Path
from google.colab import userdata

os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
Path("/content/hf_cache").mkdir(parents=True, exist_ok=True)

for name in (
    "OPENROUTER_API_KEY",
    "HF_TOKEN",
    "HUGGING_FACE_API_KEY",
    "LANGFUSE_PUBLIC_KEY",
    "LANGFUSE_SECRET_KEY",
    "LANGFUSE_BASE_URL",
):
    try:
        value = userdata.get(name)
    except Exception:
        value = ""
    if value:
        os.environ[name] = value

present = {k: bool(os.environ.get(k)) for k in ("OPENROUTER_API_KEY", "HF_TOKEN", "HUGGING_FACE_API_KEY")}
print("secrets present:", present)
if not present["OPENROUTER_API_KEY"] or not (present["HF_TOKEN"] or present["HUGGING_FACE_API_KEY"]):
    raise SystemExit("Missing OpenRouter or Hugging Face secret. Add them in Colab Secrets and rerun this cell.")

hb = Path("results/collapse_eval/heartbeat.txt")
hb.parent.mkdir(parents=True, exist_ok=True)

def _beat():
    while True:
        hb.write_text(time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()) + "\n")
        time.sleep(60)

threading.Thread(target=_beat, daemon=True).start()
print("heartbeat", hb.resolve())


## Gate (run this first)

P1 base, gen0, gen12. 20 HarmBench prompts plus full WikiText perplexity and benign diversity.

This does **not** delete the Gemma-3 base weights, so the P1 cell in this same session can reuse them.

When it finishes, open the printed summaries. Do not start P1 until those numbers have been checked.


In [ ]:
# 5. Go / no-go. Safe to rerun: finished prompts are skipped, the other 80 are not generated here.
!python scripts/run_collapse_eval.py --gate --device cuda --max-prompts 20 --max-benign 20 --generate-batch-size 2

import json
from pathlib import Path
print("\n===== GATE REPORT =====")
root = Path("results/collapse_eval/gemma3_4b_self_collapse")
for gen in ("base", "0", "12"):
    d = root / gen
    summary = json.loads((d / "summary.json").read_text()) if (d / "summary.json").is_file() else {}
    util = json.loads((d / "utility.json").read_text()) if (d / "utility.json").is_file() else {}
    tier = summary.get("tier1") or {}
    ppl = (util.get("perplexity") or {}).get("perplexity")
    div = util.get("diversity") or {}
    comp = d / "completions.jsonl"
    n_comp = sum(1 for line in comp.read_text().splitlines() if line.strip()) if comp.is_file() else 0
    sample = ""
    if comp.is_file():
        sample = (json.loads(comp.read_text().splitlines()[-1]).get("completion") or "")[:180].replace("\n", " ")
    print(f"{gen}: n_comp={n_comp} ASR={tier.get('asr')} meanSR={tier.get('mean_strongreject')} refusal={tier.get('refusal_rate')} PPL={ppl} distinct2={div.get('distinct_2')}")
    print(f"    sample: {sample}")
print("safe to disconnect — gate cell finished")


## P1 — Gemma-3-4B-it (main)

Eight checkpoints: base + gens 0, 2, 4, 6, 8, 10, 12. Full 100 prompts.

The 20 gate prompts are kept. The other 80 are generated and judged. Leave this tab open. If it dies, rerun this cell.


In [ ]:
# 6. Full P1. Resumable.
!python scripts/run_collapse_eval.py --only-lineage gemma3_4b_self_collapse --device cuda --generate-batch-size 2 --clean-cache
print("safe to disconnect — P1 cell finished")


## P2 — Gemma-2-2B Kazdan (comparison)

Six full-weight checkpoints. Each snapshot is deleted after its results are on Drive.


In [ ]:
# 7. Full P2. Resumable.
!python scripts/run_collapse_eval.py --only-lineage gemma2_2b_kazdan --device cuda --generate-batch-size 2 --clean-cache
print("safe to disconnect — P2 cell finished")


## P3 — only if P1 and P2 are on Drive and you switched this runtime to an L4 (24 GB)

Do not run this on a T4.


In [ ]:
# 8. Optional P3. L4 only.
# !python scripts/run_collapse_eval.py --only-lineage qwen25_7b_self_collapse --device cuda --generate-batch-size 1 --clean-cache
print("P3 cell is commented out on purpose")
